# Network Intrusion Detection using Machine Learning
### Dataset: CIC-IDS2017 — Canadian Institute for Cybersecurity, UNB
**Author:** Shanzay Jamil
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn

---

## What is this project?

An **Intrusion Detection System (IDS)** monitors network traffic and flags malicious activity
(attacks) versus normal (benign) traffic. In this notebook we build a **machine-learning-based IDS**:
instead of hand-written rules ("block IP X if it sends > 1000 packets/sec"), we let models *learn*
the statistical patterns that separate attack traffic from normal traffic.

**The dataset (CIC-IDS2017)** was created by the Canadian Institute for Cybersecurity at UNB. It
contains 5 days of realistic network traffic. Each row is one **network flow** (a conversation
between two machines), described by ~78 numeric features (packet sizes, timing, flag counts, etc.)
plus a **Label** column saying whether the flow was BENIGN or a specific attack (DDoS, PortScan,
Web Attack, etc.).

**Our goal:** train models that, given the features of a flow, predict *attack or benign*.

---

## 📖 Glossary — ML terms used in this notebook

| Term | Meaning |
|------|---------|
| **Feature** | An input column the model learns from (e.g., *Flow Duration*, *Packet Length Mean*). |
| **Label / Target** | The output we want to predict. Here: 0 = BENIGN, 1 = ATTACK. |
| **Training set** | The portion of data the model learns from. |
| **Test set** | Data the model has *never seen*, used to measure real-world performance. |
| **Data leakage** | When information from the test set accidentally influences training → inflated, fake performance. |
| **Stratification** | Splitting data so both train and test keep the same class proportions. |
| **Standardization** | Rescaling each feature to mean 0, std 1, so no feature dominates just because of its unit/scale. |
| **Overfitting** | Model memorizes training data instead of learning general patterns → great on train, poor on new data. |
| **Accuracy** | % of predictions that are correct. Misleading on imbalanced data! |
| **Precision** | Of the flows we flagged as ATTACK, how many really were attacks? (Low precision = false alarms.) |
| **Recall** | Of all real attacks, how many did we catch? (Low recall = missed intrusions — the worst failure for an IDS.) |
| **F1-score** | Harmonic mean of precision & recall — a single balanced number. |
| **Confusion matrix** | 2×2 table of correct/incorrect predictions per class. |
| **Random Forest** | An *ensemble* of many decision trees, each trained on random subsets of rows/features; they vote. Robust, handles raw scales, gives feature importances. |
| **MLP (Multi-Layer Perceptron)** | A basic feed-forward neural network: layers of neurons with learned weights, trained by backpropagation. Needs scaled inputs. |


## Step 1 — Import Libraries

**Why:** Load every tool before use so a missing package fails fast, at the top.

- `pandas` — tabular data handling (DataFrames)
- `numpy` — fast numeric arrays (also gives us `np.inf` for cleaning)
- `matplotlib` / `seaborn` — plotting
- `sklearn.model_selection.train_test_split` — split data into train/test
- `sklearn.preprocessing.StandardScaler` — feature standardization (for the neural network)
- `RandomForestClassifier`, `MLPClassifier` — our two models
- `sklearn.metrics` — accuracy, precision/recall/F1 report, confusion matrix

> 🔧 **CHANGE #1:** Removed `warnings.filterwarnings('ignore')`.
> Blanket-suppressing warnings is dangerous — it was hiding a real problem
> (the neural network was not converging; see Step 5). Let warnings speak.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, f1_score, recall_score,
                             precision_score)

print('Libraries loaded successfully')

## Step 2 — Load Dataset

**Why:** CIC-IDS2017 ships as one CSV per capture day/session. We load several days so the model
sees *multiple attack types* (DoS, DDoS, PortScan, Web Attacks) plus benign traffic, then
concatenate them into one DataFrame.

**What the code does:**
1. Read each CSV (`low_memory=False` avoids mixed-dtype guessing errors on big files).
2. `str.strip()` the column names — this dataset infamously has a leading space in `' Label'`.
3. Concatenate all days into a single DataFrame.

> 🔧 **CHANGE #2 — fixed file names.** The official CIC-IDS2017 files use a **dot** before
> `pcap`, e.g. `Wednesday-workingHours.pcap_ISCX.csv` — the previous version used an underscore
> (`_pcap_ISCX.csv`) and would crash with `FileNotFoundError` on the real files.
>
> 🔧 **CHANGE #3 — smarter row sampling.** Previously `nrows=50000` read only the *first* 50k rows
> of each file. CSV rows are roughly in time order, so this can grab mostly-BENIGN morning traffic
> and miss the attacks that happen later in the day. Now we read the full file and take a
> **random sample** (reproducible via `random_state=42`), which preserves each file's true
> benign/attack mix. If your machine has enough RAM, set `SAMPLE_PER_FILE = None` to use everything.

In [ ]:
# Official CIC-IDS2017 file names (note the '.pcap_ISCX.csv' with a dot)
files = [
    'data/Tuesday-WorkingHours.pcap_ISCX.csv',
    'data/Wednesday-workingHours.pcap_ISCX.csv',
    'data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
]

SAMPLE_PER_FILE = 50000   # set to None to load every row (needs more RAM)

dfs = []
for f in files:
    temp = pd.read_csv(f, low_memory=False)
    temp.columns = temp.columns.str.strip()          # fix ' Label' -> 'Label'
    if SAMPLE_PER_FILE is not None and len(temp) > SAMPLE_PER_FILE:
        # Random sample keeps the file's real benign/attack proportions,
        # unlike nrows= which only reads the top of the file.
        temp = temp.sample(n=SAMPLE_PER_FILE, random_state=42)
    dfs.append(temp)
    print(f'{f}: {temp.shape[0]} rows loaded')

df = pd.concat(dfs, ignore_index=True)
print(f'\nDataset shape: {df.shape}')
print(f'\nLabel distribution:')
print(df['Label'].value_counts())

## Step 3 — Exploratory Data Analysis (EDA)

**Why:** Before modelling, *look at the data*. The single most important thing to check for a
classification problem is **class balance** — how many samples of each label we have.

If 95% of flows are BENIGN, a useless model that always predicts "BENIGN" scores 95% accuracy.
Knowing the imbalance tells us (a) to use `stratify` when splitting, and (b) that accuracy alone
is not a trustworthy metric — we must also look at per-class precision/recall (Step 6).

In [ ]:
label_counts = df['Label'].value_counts()

plt.figure(figsize=(12,6))
colors = ['#2ecc71' if l == 'BENIGN' else '#e74c3c' for l in label_counts.index]
label_counts.plot(kind='bar', color=colors)
plt.title('Traffic Distribution — CIC-IDS2017', fontsize=14, fontweight='bold')
plt.xlabel('Traffic Type')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('traffic_distribution.png', dpi=150)
plt.show()

print('Class imbalance check:')
print(f"BENIGN share: {(df['Label']=='BENIGN').mean()*100:.1f}% of all flows")

## Step 4 — Data Preprocessing

**Why each sub-step exists:**

1. **Replace ±infinity with NaN, then drop NaN rows.** Flow-rate features like `Flow Bytes/s`
   contain `inf` when a flow's duration is 0 (division by zero during dataset generation).
   Models cannot train on `inf`/`NaN`.

2. **Drop duplicate rows** *(new — see Change #4)*. CIC-IDS2017 contains many exact duplicate
   flows. If identical rows land in both train **and** test, the model is being "tested" on rows
   it memorized — a form of **data leakage** that inflates accuracy. Removing duplicates makes
   the evaluation honest (expect accuracy to drop slightly — that's a *good* sign).

3. **Binary label encoding.** We map BENIGN → 0 and every attack type → 1. This turns a
   multi-class problem into simpler **binary classification** ("is this flow an attack?").
   Detecting *which* attack is a natural next step (see end of notebook).

4. **Keep only numeric feature columns.** `select_dtypes(include=[np.number])` drops text columns
   the models can't use directly.

5. **Train/test split (80/20), stratified** *(new — see Change #5)*. We hold out 20% of the data
   the models never see, to estimate real-world performance. `stratify=y` guarantees the test set
   has the same attack/benign ratio as the training set — essential with imbalanced classes,
   otherwise a rare attack type could end up entirely missing from one side of the split.

6. **Standardize features — for the neural network only.**
   `StandardScaler` rescales each feature to mean 0, std 1. Neural networks train poorly when one
   feature ranges 0–1 and another 0–10,000,000 (huge features dominate the gradients).
   **Crucially, the scaler is `fit` on the training set only** and merely applied (`transform`)
   to the test set — fitting on all data would leak test-set statistics into training.

> 🔧 **CHANGE #4:** Added `drop_duplicates()` — removes train/test contamination.
> 🔧 **CHANGE #5:** Added `stratify=y` to `train_test_split`.
> 🔧 **CHANGE #6:** Random Forest now trains on the **unscaled** data. Tree-based models split on
> thresholds ("is Flow Duration > 5000?"), so rescaling changes nothing for them — scaling is kept
> only where it matters, for the MLP. This also keeps feature-importance values in the original,
> interpretable units.

In [ ]:
# 1. Remove infinite and null values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 2. Remove duplicate rows (prevents train/test leakage)
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows ({(before-len(df))/before*100:.1f}%)')

# 3. Convert to binary label: BENIGN=0, Attack=1
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
print(f'Benign samples: {(df["Label"]==0).sum()}')
print(f'Attack samples: {(df["Label"]==1).sum()}')

# 4. Features (numeric only) and target
X = df.drop('Label', axis=1).select_dtypes(include=[np.number])
y = df['Label']

# 5. Stratified train/test split (keeps class proportions identical in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Standardize features — needed for the neural network only.
#    fit on TRAIN only, transform TEST with the same parameters (no leakage).
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')
print(f'Number of features: {X.shape[1]}')

## Step 5 — Train Models

We train two very different model families and compare them.

### Model A — Random Forest

**How it works:** builds `n_estimators=100` decision trees. Each tree is trained on a random
bootstrap sample of the rows and considers a random subset of features at each split
(this randomness is why it's called a *random* forest). At prediction time all trees vote.

**Why it suits this problem:** robust to outliers and unscaled features, handles the ~78 mixed-scale
flow features natively, rarely overfits badly with enough trees, and provides
**feature importances** — which we use in Step 6 to see *which* traffic characteristics reveal attacks.

`n_jobs=-1` = use all CPU cores. `random_state=42` = reproducible results.

> 🔧 **CHANGE #6 (applied here):** RF is fit on `X_train` (raw values), not the scaled copy.

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)                      # raw features — trees don't need scaling
rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1  = f1_score(y_test, rf_pred)
print(f'Random Forest Accuracy: {rf_acc*100:.2f}%  |  F1: {rf_f1:.4f}')
print(classification_report(y_test, rf_pred, target_names=['BENIGN','ATTACK']))

### Model B — Neural Network (MLP)

**How it works:** a Multi-Layer Perceptron with two hidden layers (64 then 32 neurons). Each neuron
computes a weighted sum of its inputs and applies a non-linear activation (ReLU). Training uses
**backpropagation**: the network's prediction error is propagated backwards to nudge every weight
in the direction that reduces the error, over many passes (**epochs**) through the data.

**Convergence** means the training loss has stopped improving — the network has finished learning.

> 🔧 **CHANGE #7 — the most important fix.** The old code used `max_iter=50`, which is far too few
> epochs: scikit-learn raises a `ConvergenceWarning` ("Maximum iterations reached and the
> optimization hasn't converged"), but Change #1's warning suppression was hiding it, so the model
> was silently under-trained. Now:
> - `max_iter=300` — enough epochs to converge, and
> - `early_stopping=True` — holds out 10% of training data as a validation set and stops
>   automatically once validation score stops improving. This both prevents wasted epochs **and**
>   protects against overfitting.

In [ ]:
print('Training Neural Network...')
nn = MLPClassifier(hidden_layer_sizes=(64, 32),
                   max_iter=300,
                   early_stopping=True,      # stop when validation score plateaus
                   n_iter_no_change=10,
                   random_state=42)
nn.fit(X_train_s, y_train)                   # scaled features — NNs need them
nn_pred = nn.predict(X_test_s)

nn_acc = accuracy_score(y_test, nn_pred)
nn_f1  = f1_score(y_test, nn_pred)
print(f'Converged after {nn.n_iter_} epochs')
print(f'Neural Network Accuracy: {nn_acc*100:.2f}%  |  F1: {nn_f1:.4f}')
print(classification_report(y_test, nn_pred, target_names=['BENIGN','ATTACK']))

## Step 6 — Results & Visualizations

**Why we report more than accuracy:** with imbalanced classes, accuracy hides failure on the
minority class. For an IDS the metric that matters most is **recall on the ATTACK class** —
the fraction of real attacks we actually catch. A missed attack (false negative) is far more
costly than a false alarm (false positive).

> 🔧 **CHANGE #8:** The comparison now plots **Accuracy, Attack-Recall, and F1** side by side,
> and the y-axis starts at 0 — the old chart's `ylim(80,100)` visually exaggerated a tiny
> difference between models, which reads as misleading to a technical audience.

In [ ]:
# Compute the three metrics for both models
metrics = {
    'Random Forest':  [rf_acc, recall_score(y_test, rf_pred), rf_f1],
    'Neural Network': [nn_acc, recall_score(y_test, nn_pred), nn_f1],
}
metric_names = ['Accuracy', 'Attack Recall', 'F1-score']

x = np.arange(len(metric_names))
width = 0.35
plt.figure(figsize=(9,5))
for i, (model, vals) in enumerate(metrics.items()):
    bars = plt.bar(x + i*width, [v*100 for v in vals], width,
                   label=model, color=['#3498db', '#2ecc71'][i])
    for bar, v in zip(bars, vals):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                 f'{v*100:.2f}', ha='center', fontsize=9, fontweight='bold')
plt.xticks(x + width/2, metric_names)
plt.ylabel('Score (%)')
plt.ylim(0, 105)                       # honest axis, starts at zero
plt.title('Model Comparison — CIC-IDS2017', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

### Confusion Matrix — Random Forest

**How to read it:** rows = truth, columns = prediction.
- Top-left: benign flows correctly passed (true negatives)
- Bottom-right: attacks correctly caught (true positives)
- **Bottom-left: attacks we MISSED (false negatives) — the number an IDS most wants at zero**
- Top-right: false alarms (false positives) — these waste analyst time

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BENIGN','ATTACK'],
            yticklabels=['BENIGN','ATTACK'])
plt.title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Missed attacks (false negatives): {fn}')
print(f'False alarms   (false positives): {fp}')

### Feature Importance

**Why:** Random Forest measures how much each feature reduced prediction error across all trees.
This tells us *which traffic characteristics actually reveal attacks* — valuable both for
interpretability and, later, for building faster models that use only the top features
(**feature selection**).

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).nlargest(15)
plt.figure(figsize=(10,6))
feat_imp.sort_values().plot(kind='barh', color='#3498db')
plt.title('Top 15 Most Important Features — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## Summary

*(Re-run the notebook and paste your actual numbers here — after removing duplicates,
expect slightly lower but **honest** scores.)*

| Model | Accuracy | Attack Recall | F1 |
|-------|----------|---------------|----|
| Random Forest | _fill in_ | _fill in_ | _fill in_ |
| Neural Network | _fill in_ | _fill in_ | _fill in_ |

**Key findings:**
- Random Forest handles this tabular flow data extremely well with zero tuning.
- Flow timing (IAT = Inter-Arrival Time), duration, and packet-length statistics are the most
  discriminative features.
- Volumetric attacks (DDoS, PortScan) are easiest to detect — their traffic shape is very distinct.
- Web attacks (XSS, SQL injection) are hardest — few samples and they hide inside normal-looking HTTP flows.

---

## 🔧 Changelog — what was fixed vs. the original notebook

| # | Where | Change | Why |
|---|-------|--------|-----|
| 1 | Step 1 | Removed `warnings.filterwarnings('ignore')` | It was hiding the MLP ConvergenceWarning (a real bug) |
| 2 | Step 2 | File names corrected to `.pcap_ISCX.csv` (dot, not underscore) | Original paths don't match the official dataset files → `FileNotFoundError` |
| 3 | Step 2 | `nrows=50000` → random `sample(n=50000)` per file | Top-of-file rows are time-ordered and skew toward BENIGN; random sampling keeps true class mix |
| 4 | Step 4 | Added `drop_duplicates()` | Duplicate rows in train **and** test = leakage = inflated accuracy |
| 5 | Step 4 | Added `stratify=y` to the split | Keeps attack/benign ratio identical in train & test with imbalanced data |
| 6 | Step 4–5 | RF trains on raw (unscaled) features | Trees are scale-invariant; scaling kept only for the MLP where it matters |
| 7 | Step 5 | MLP `max_iter=50` → `300` + `early_stopping=True` | Old model never converged (silently under-trained) |
| 8 | Step 6 | Added Attack-Recall & F1 to the comparison; y-axis starts at 0 | Accuracy alone is misleading on imbalanced data; truncated axes exaggerate differences |

---

## 🚀 What to do next (roadmap)

1. **Multi-class classification** — predict the *specific* attack type instead of binary
   attack/benign: keep the original string labels and let the models output 15 classes.
   Report a per-class confusion matrix to see which attacks get confused with each other.

2. **Handle class imbalance explicitly** — try `class_weight='balanced'` in RandomForest, or
   oversample minority attacks with **SMOTE** (`imblearn` library). Measure the effect on
   Web-Attack recall specifically.

3. **Cross-validation** — replace the single 80/20 split with 5-fold **stratified
   cross-validation** (`StratifiedKFold`) to get a mean ± std for each metric instead of one
   number that depends on split luck.

4. **Stronger models** — benchmark **XGBoost / LightGBM** (gradient-boosted trees), which usually
   beat Random Forest on tabular data.

5. **Feature selection** — retrain using only the top-15 important features; if performance barely
   drops, you have a much faster model suitable for real-time detection.

6. **Cross-day generalization test** — train on some days, test on a *different* day's file.
   This is far more realistic than random splitting (real IDSs face traffic they've never seen)
   and is the kind of evaluation CIC researchers respect.

7. **Deploy a demo** — wrap the trained RF in a small script that reads flow features from
   a CSV (or CICFlowMeter output) and prints alerts.